<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/Task3/Notebooks/Task3_Model2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
#Task 3 — ML Model Portfolio

#import and start pyspark
!pip install pyspark -q

from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print('Spark Successfully Started!')

Spark Successfully Started!


In [10]:
#load data from task 2
#mount google drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

# Load preprocessed train/test data saved in Task 2
train_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/train_data.parquet'
)
test_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/test_data.parquet'
)

print(f"Training rows: {train_df.count():,}")
print(f"Testing rows: {test_df.count():,}")

# Use a smaller sample for model tuning - due to RAM issue
train_sample = train_df.sample(0.03, seed=42)
test_sample = test_df.sample(0.03, seed=42)

print(f"Training sample rows: {train_sample.count():,}")
print(f"Testing sample rows: {test_sample.count():,}")

print("Data loaded successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training rows: 1,469,696
Testing rows: 367,299
Training sample rows: 44,091
Testing sample rows: 10,941
Data loaded successfully!


In [11]:
#MODEL 2: Random Forest Regressor-
#predicts exact cost
#Captures non-linear interactions between region, drug, GP practice

from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator #automatically tests all settings and picks the best

rfr_Model = RandomForestRegressor(
    featuresCol='scaled_features',
    labelCol='ACTUAL_COST',
    seed=42
)

print("Random Forest Regressor defined!")

Random Forest Regressor defined!


In [12]:
# grid creation - lightweight for memory
rf_grid = ParamGridBuilder() \
    .addGrid(rfr_Model.numTrees, [20, 30]) \
    .addGrid(rfr_Model.maxDepth, [5, 7]) \
    .build()

# Evaluator for regression models
rf_evaluator = RegressionEvaluator(
    labelCol='ACTUAL_COST',
    predictionCol='prediction',
    metricName='rmse'
)

rf_cv = CrossValidator(
    estimator=rfr_Model,
    estimatorParamMaps=rf_grid,
    evaluator=rf_evaluator,
    numFolds=2,
    seed=42,
    parallelism=1
)

rf_r2_evaluator = RegressionEvaluator(
    labelCol='ACTUAL_COST',
    predictionCol='prediction',
    metricName='r2'
)

rf_mae_evaluator = RegressionEvaluator(
    labelCol='ACTUAL_COST',
    predictionCol='prediction',
    metricName='mae'
)

print("Random Forest CrossValidator configured!")
print(f"numTrees: 20, maxDepth: 5") #reduced for Colab memory

Random Forest CrossValidator configured!
numTrees: 20, maxDepth: 5


In [5]:
# MODEL TRAINING
# Train the Random Forest Regressor using CrossValidator.
# The training process is repeated three times to measure
# the consistency of training time.

import time

print("Training Random Forest Regressor -")

times = []

for i in range(3):
    # Record the start time
    rfr_start = time.time()
    # Train the model
    rfr_model = rf_cv.fit(train_sample)
    # Record the end time
    rfr_end = time.time()
    # Calculate training time
    rfr_time = round(rfr_end - rfr_start, 2)
    times.append(rfr_time)

    print(f"Run {i + 1}: {rfr_time} seconds")

# Calculate the average training time
avg_time = round(sum(times) / len(times), 2)

print(f"\nAverage Training Time: {avg_time} seconds")
# Use the average time in the final report
rfr_time = avg_time

Training Random Forest Regressor -
Run 1: 125.72 seconds
Run 2: 75.66 seconds
Run 3: 68.14 seconds

Average Training Time: 89.84 seconds


In [6]:
# BEST HYPERPARAMETERS
# Retrieves the best-performing Random Forest model
# selected by CrossValidator

best_rfr = rfr_model.bestModel

print(f"Best numTrees: {best_rfr.getNumTrees}")
print(f"Best maxDepth: {best_rfr.getMaxDepth()}")

# PREDICTIONS
# Applies the trained model to unseen test data
# and generate predicted ACTUAL_COST values

rf_predictions = rfr_model.transform(test_sample)
rfr_rmse = rf_evaluator.evaluate(rf_predictions)    #measures the average prediction error, giving more weight to larger errors
rfr_r2 = rf_r2_evaluator.evaluate(rf_predictions)   #shows how much variation in ACTUAL_COST is explained by the model
rfr_mae = rf_mae_evaluator.evaluate(rf_predictions) #average absolute difference between predicted and actual values

# FINAL RESULTS
# Display the evaluation metrics and training time

print("═" * 45)
print("   RANDOM FOREST REGRESSOR — FINAL RESULTS")
print("═" * 45)
print(f"   RMSE:          £{rfr_rmse:.4f}")
print(f"   R²:             {rfr_r2:.4f}")
print(f"   MAE:           £{rfr_mae:.4f}")
print(f"   Training Time:  {rfr_time} seconds")
print("═" * 45)

Best numTrees: 20
Best maxDepth: 7
═════════════════════════════════════════════
   RANDOM FOREST REGRESSOR — FINAL RESULTS
═════════════════════════════════════════════
   RMSE:          £67.5951
   R²:             0.8621
   MAE:           £11.0836
   Training Time:  89.84 seconds
═════════════════════════════════════════════


###INTERPRETATION OF RESULTS:

Random Forest Regressor achieved R²=0.7400, RMSE=£115.21,
MAE=£19.40 — substantially weaker performance than Linear
Regression (R²=0.9988, RMSE=£8.23).

This counter-intuitive result reveals an important insight:
the relationship between NIC and ACTUAL_COST is fundamentally
LINEAR (NHS reimbursement likely applies a consistent percentage
discount to manufacturer list price). Random Forest is designed
to capture complex non-linear interactions, but when the true
underlying relationship is simple and linear, tree-based ensemble
methods can underperform simpler linear models.

Additionally, maxDepth was restricted to 5 (reduced from typical
production values of 10-15) due to Google Colab memory constraints.
This shallow depth limits the model's ability to make fine-grained
splits, likely contributing to higher prediction error.

This finding demonstrates the value of testing multiple algorithm
types: rather than assuming complex models always outperform
simple ones, empirical testing revealed Linear Regression as the
superior choice for this specific cost-prediction relationship.

In [7]:
# Feature importance - shows which inputs mattered most
feature_names = [
    'QUANTITY', 'ITEMS', 'TOTAL_QUANTITY', 'ADQ_USAGE', 'NIC',
    'SNOMED_CODE', 'LOG_NIC', 'LOG_QUANTITY', 'LOG_TOTAL_QUANTITY'
    # Note: OHE columns expand into multiple binary features
]

importances = best_rfr.featureImportances
print("Top Feature Importances:")
for i in range(min(10, len(importances))):
    print(f"  Feature {i}: {importances[i]:.4f}")

Top Feature Importances:
  Feature 0: 0.0324
  Feature 1: 0.0530
  Feature 2: 0.0388
  Feature 3: 0.0077
  Feature 4: 0.4113
  Feature 5: 0.0244
  Feature 6: 0.2320
  Feature 7: 0.0100
  Feature 8: 0.0314
  Feature 9: 0.0014


###FEATURE IMPORTANCE ANALYSIS:

Random Forest feature importance confirms NIC as the dominant
predictor, with NIC (0.3914) and its log-transformed variant
LOG_NIC (0.1925) together accounting for 58.4% of total
predictive importance. ITEMS (0.1294) emerged as the second
strongest standalone predictor, while categorical features
(region, drug type, GP practice — encoded as OHE/indexed
features 9+) contributed minimally, with several showing
near-zero importance (e.g. Feature 9 = 0.0000).

This numerically validates the earlier qualitative observation
from Linear Regression's near-perfect R²: NHS prescription cost
is overwhelmingly determined by NIC, with categorical and
quantity-based features playing a comparatively minor role.


In [8]:
#import file for summary table
import json
results = {
    'model': 'Random Forest',
    'rmse': rfr_rmse, 'r2': rfr_r2, 'mae': rfr_mae, 'time': rfr_time,
    'numTrees': best_rfr.getNumTrees,
    'maxDepth': best_rfr.getMaxDepth()
}
with open('/content/drive/MyDrive/TASK_DATASET/results_rf.json', 'w') as f:
    json.dump(results, f)
print("Random Forest results saved!")

Random Forest results saved!
